# Advanced Problems with Solutions: `__getattr__`, `__setattr__`, and Dynamic Data Models

This notebook develops the topic from a dictionary-backed "database" into a small, robust dynamic object layer.

## Learning goals

By the end, you should be able to:

- explain exactly when `__getattr__` is invoked;
- use `object.__getattribute__` and `object.__setattr__` to avoid accidental recursion;
- distinguish internal attributes from dynamic database fields;
- validate writes safely;
- handle attribute/method name collisions;
- expose useful names through `__dir__`;
- track dirty fields and changes;
- support commit/rollback behavior;
- resolve foreign-key relationships dynamically;
- add computed fields without corrupting precedence rules;
- add caching with correct invalidation;
- serialize dynamic records safely;
- build a compact mini-ORM using these ideas.

> **Best-practice rule:** override `__getattr__` when fallback lookup is enough. Override `__getattribute__` only when you truly need to intercept *every* attribute access.

## Reference dataset

We will use a plain dictionary as a deterministic stand-in for a database. This keeps the focus on Python's data model rather than SQL or an ORM package.

In [1]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from typing import Any, Callable, Mapping
import json

DB = {
    "Person": {
        1: {"first_name": "Isaac", "last_name": "Newton", "born": 1642, "country_id": 1},
        2: {"first_name": "Gottfried", "last_name": "von Leibniz", "born": 1646, "country_id": 5},
        3: {"first_name": "Joseph", "last_name": "Fourier", "born": 1768, "country_id": 3},
        4: {"first_name": "Bernhard", "last_name": "Riemann", "born": 1826, "country_id": 5},
        5: {"first_name": "David", "last_name": "Hilbert", "born": 1862, "country_id": 5},
        6: {"first_name": "Srinivasa", "last_name": "Ramanujan", "born": 1887, "country_id": 4},
        7: {"first_name": "John", "last_name": "von Neumann", "born": 1903, "country_id": 2},
        8: {"first_name": "Andrew", "last_name": "Wiles", "born": 1928, "country_id": 6},
    },
    "Country": {
        1: {"name": "United Kingdom", "capital": "London", "continent": "Europe"},
        2: {"name": "Hungary", "capital": "Budapest", "continent": "Europe"},
        3: {"name": "France", "capital": "Paris", "continent": "Europe"},
        4: {"name": "India", "capital": "New Delhi", "continent": "Asia"},
        5: {"name": "Germany", "capital": "Berlin", "continent": "Europe"},
        6: {"name": "USA", "capital": "Washington DC", "continent": "North America"},
    },
}

ORIGINAL_DB = deepcopy(DB)
print("Tables:", tuple(DB))
print("Person fields:", tuple(DB["Person"][1]))

Tables: ('Person', 'Country')
Person fields: ('first_name', 'last_name', 'born', 'country_id')


## Example 1 — Why `__getattr__` is a fallback hook

`__getattr__` is called only after normal attribute lookup fails. That makes it safer than `__getattribute__` for many dynamic-field designs.

In [2]:
class LookupDemo:
    class_attribute = "found normally"

    def __init__(self):
        self.instance_attribute = "also found normally"

    def __getattr__(self, name):
        print(f"__getattr__ fallback for {name!r}")
        return f"<dynamic:{name}>"

demo = LookupDemo()

print(demo.class_attribute)       # no __getattr__
print(demo.instance_attribute)    # no __getattr__
print(demo.missing_attribute)     # __getattr__ runs

found normally
also found normally
__getattr__ fallback for 'missing_attribute'
<dynamic:missing_attribute>


## Problem 1 — Build a safe read-only dynamic record

Implement `ReadOnlyRecord` so dictionary keys can be read as attributes.

### Requirements

1. Store the mapping internally as `_record`.
2. Use `object.__setattr__` during initialization.
3. Return a field value from `__getattr__`.
4. Raise `AttributeError`, not `KeyError`, for unknown attributes.
5. Do not override `__getattribute__`.

Why `AttributeError`? Python tooling, `hasattr`, descriptors, and many introspection features expect missing attributes to fail with `AttributeError`.

In [3]:
# SOLUTION 1

class ReadOnlyRecord:
    def __init__(self, record: Mapping[str, Any]):
        object.__setattr__(self, "_record", dict(record))

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        try:
            return record[name]
        except KeyError:
            raise AttributeError(f"No dynamic field {name!r}") from None

p = ReadOnlyRecord(DB["Person"][1])

assert p.first_name == "Isaac"
assert p.born == 1642
assert hasattr(p, "last_name")
assert not hasattr(p, "does_not_exist")

print(p.first_name, p.last_name, p.born)

Isaac Newton 1642


## Problem 2 — Trace an accidental recursion bug

The implementation below looks plausible but is broken:

```python
class BrokenRecord:
    def __init__(self, record):
        self._record = record

    def __setattr__(self, name, value):
        if name in self._record:
            self._record[name] = value
        else:
            raise AttributeError(name)
```

Explain why constructing `BrokenRecord(...)` can recurse, then repair the design.

In [4]:
# SOLUTION 2

class SafeWritableRecord:
    def __init__(self, record: Mapping[str, Any]):
        # Bypass our custom __setattr__ while bootstrapping internal state.
        object.__setattr__(self, "_record", dict(record))

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]
        raise AttributeError(name)

    def __setattr__(self, name: str, value: Any) -> None:
        record = object.__getattribute__(self, "_record")
        if name in record:
            record[name] = value
            return
        raise AttributeError(f"Unknown field {name!r}")

r = SafeWritableRecord({"x": 1, "y": 2})
r.x = 10

assert r.x == 10
assert r.y == 2
print(r._record)

{'x': 10, 'y': 2}


### Why the broken version recurses

During `__init__`, the assignment `self._record = record` immediately invokes the custom `__setattr__`. That implementation then evaluates `self._record` before `_record` exists. Normal lookup fails and may trigger other hooks, repeatedly re-entering the dynamic machinery.

The robust pattern is to use `object.__setattr__(self, "_record", ...)` for trusted internal state.

## Problem 3 — Separate internal attributes from dynamic fields

A real class often needs internal state such as `_dirty`, `_schema`, `_table`, or `_session`.

Implement a rule:

- names starting with `_` are internal and can be stored on the instance;
- names without `_` must already exist as database fields;
- unknown public fields must be rejected.

In [5]:
# SOLUTION 3

class ManagedRecord:
    def __init__(self, record: Mapping[str, Any]):
        object.__setattr__(self, "_record", dict(record))
        object.__setattr__(self, "_dirty", set())

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]
        raise AttributeError(name)

    def __setattr__(self, name: str, value: Any) -> None:
        if name.startswith("_"):
            object.__setattr__(self, name, value)
            return

        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise AttributeError(f"Unknown public field {name!r}")

        old = record[name]
        record[name] = value

        if old != value:
            dirty = object.__getattribute__(self, "_dirty")
            dirty.add(name)

record = ManagedRecord({"name": "Ada", "year": 1815})
record.year = 1816
record._note = "internal metadata"

assert record.year == 1816
assert record._dirty == {"year"}
assert record._note == "internal metadata"

print("record:", record._record)
print("dirty:", record._dirty)

record: {'name': 'Ada', 'year': 1816}
dirty: {'year'}


## Problem 4 — Add schema validation without cluttering `__setattr__`

Best practice: keep special methods short. Move field-specific validation into ordinary helper functions.

Build a schema where each field maps to a validator. Reject invalid writes *before* mutating the record.

In [6]:
# SOLUTION 4

Validator = Callable[[Any], None]

def validate_non_empty_string(value: Any) -> None:
    if not isinstance(value, str) or not value.strip():
        raise ValueError("expected a non-empty string")

def validate_year(value: Any) -> None:
    if not isinstance(value, int):
        raise TypeError("year must be int")
    if not (0 <= value <= 3000):
        raise ValueError("year out of allowed range")

def validate_positive_int(value: Any) -> None:
    if not isinstance(value, int):
        raise TypeError("expected int")
    if value <= 0:
        raise ValueError("expected positive int")

PERSON_SCHEMA: dict[str, Validator] = {
    "first_name": validate_non_empty_string,
    "last_name": validate_non_empty_string,
    "born": validate_year,
    "country_id": validate_positive_int,
}

class ValidatedRecord(ManagedRecord):
    def __init__(self, record: Mapping[str, Any], schema: Mapping[str, Validator]):
        super().__init__(record)
        object.__setattr__(self, "_schema", dict(schema))

    def __setattr__(self, name: str, value: Any) -> None:
        if name.startswith("_"):
            object.__setattr__(self, name, value)
            return

        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise AttributeError(name)

        schema = object.__getattribute__(self, "_schema")
        validator = schema.get(name)
        if validator is not None:
            validator(value)

        super().__setattr__(name, value)

v = ValidatedRecord(DB["Person"][1], PERSON_SCHEMA)
v.first_name = "Isaac"
v.born = 1643

try:
    v.born = "1643"
except TypeError as exc:
    print("Rejected:", exc)

assert v.born == 1643

Rejected: year must be int


## Problem 5 — Handle field/method name collisions

Suppose a database record contains fields named `fields`, `save`, or `to_dict`, while the Python class also defines methods with those names.

Normal lookup wins before `__getattr__`, so those field values would become inaccessible through `obj.fields`, `obj.save`, etc.

Design a collision-safe API with:

- attribute access for non-conflicting dynamic fields;
- explicit `get_field(name)` for every field;
- explicit `set_field(name, value)` for every field.

In [7]:
# SOLUTION 5

class CollisionSafeRecord(ManagedRecord):
    def fields(self) -> tuple[str, ...]:
        return tuple(object.__getattribute__(self, "_record"))

    def get_field(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        try:
            return record[name]
        except KeyError:
            raise KeyError(f"Unknown field {name!r}") from None

    def set_field(self, name: str, value: Any) -> None:
        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise KeyError(f"Unknown field {name!r}")
        old = record[name]
        record[name] = value
        if old != value:
            object.__getattribute__(self, "_dirty").add(name)

collision = CollisionSafeRecord({
    "name": "example",
    "fields": "DATABASE VALUE",
    "save": "DATABASE VALUE TOO",
})

print("method call:", collision.fields())
print("colliding field:", collision.get_field("fields"))
print("save field:", collision.get_field("save"))

assert collision.name == "example"
assert collision.get_field("fields") == "DATABASE VALUE"

method call: ('name', 'fields', 'save')
colliding field: DATABASE VALUE
save field: DATABASE VALUE TOO


### Best practice for collisions

Do **not** switch to `__getattribute__` merely to force dynamic fields to win. That can hide methods, properties, descriptors, and class attributes. An explicit field API is easier to reason about.

## Problem 6 — Improve discoverability with `__dir__`

Dynamic attributes are often invisible to tab completion.

Implement `__dir__` so the record's current dynamic field names are included while preserving normal attributes and methods.

In [8]:
# SOLUTION 6

class DiscoverableRecord(CollisionSafeRecord):
    def __dir__(self):
        normal_names = set(super().__dir__())
        field_names = set(object.__getattribute__(self, "_record"))
        return sorted(normal_names | field_names)

d = DiscoverableRecord({"alpha": 1, "beta": 2})
names = dir(d)

assert "alpha" in names
assert "beta" in names
assert "get_field" in names

print([name for name in names if name in {"alpha", "beta", "get_field", "set_field"}])

['alpha', 'beta', 'get_field', 'set_field']


## Problem 7 — Track changes, not only dirty names

Create a record that remembers each field's original value and current value.

Requirements:

- a field changed multiple times still reports the *first* original value;
- if changed back to the original value, it disappears from the change set;
- expose a read-only copy through `changes`.

In [9]:
# SOLUTION 7

class ChangeTrackingRecord(ManagedRecord):
    def __init__(self, record: Mapping[str, Any]):
        super().__init__(record)
        object.__setattr__(self, "_original", deepcopy(dict(record)))

    @property
    def changes(self) -> dict[str, tuple[Any, Any]]:
        record = object.__getattribute__(self, "_record")
        original = object.__getattribute__(self, "_original")
        return {
            name: (original[name], record[name])
            for name in record
            if record[name] != original[name]
        }

    def __setattr__(self, name: str, value: Any) -> None:
        super().__setattr__(name, value)
        if not name.startswith("_"):
            dirty = object.__getattribute__(self, "_dirty")
            if name not in self.changes:
                dirty.discard(name)

c = ChangeTrackingRecord({"x": 1, "y": 2})
c.x = 10
c.x = 20

assert c.changes == {"x": (1, 20)}

c.x = 1
assert c.changes == {}
assert c._dirty == set()

print("changes after reverting:", c.changes)

changes after reverting: {}


## Problem 8 — Add commit and rollback semantics

Turn change tracking into a tiny unit-of-work model.

- `commit()` accepts current values as the new baseline.
- `rollback()` restores original values.
- both operations clear dirty state.

In [10]:
# SOLUTION 8

class TransactionalRecord(ChangeTrackingRecord):
    def commit(self) -> None:
        record = object.__getattribute__(self, "_record")
        object.__setattr__(self, "_original", deepcopy(record))
        object.__getattribute__(self, "_dirty").clear()

    def rollback(self) -> None:
        original = object.__getattribute__(self, "_original")
        object.__setattr__(self, "_record", deepcopy(original))
        object.__getattribute__(self, "_dirty").clear()

t = TransactionalRecord({"name": "old", "count": 1})
t.name = "new"
t.count = 2

assert t.changes == {"name": ("old", "new"), "count": (1, 2)}

t.rollback()
assert t.name == "old"
assert t.count == 1
assert t.changes == {}

t.name = "accepted"
t.commit()

assert t.name == "accepted"
assert t.changes == {}
print(t._record)

{'name': 'accepted', 'count': 1}


## Problem 9 — Build a callable table object

Create `Table` so this syntax works:

```python
people = Table(DB, "Person")
person = people(1)
```

The returned object should be a `TransactionalRecord`. Add:

- `table_name`;
- `__len__`;
- `__contains__`;
- clear exceptions for unknown tables and unknown record IDs.

In [11]:
# SOLUTION 9

class Table:
    def __init__(self, db: Mapping[str, Mapping[int, dict[str, Any]]], table_name: str):
        if table_name not in db:
            raise ValueError(f"Unknown table {table_name!r}")
        self._db = db
        self._table_name = table_name

    @property
    def table_name(self) -> str:
        return self._table_name

    def __len__(self) -> int:
        return len(self._db[self._table_name])

    def __contains__(self, record_id: int) -> bool:
        return record_id in self._db[self._table_name]

    def __call__(self, record_id: int) -> TransactionalRecord:
        table = self._db[self._table_name]
        if record_id not in table:
            raise KeyError(
                f"Record id {record_id!r} not found in table {self._table_name!r}"
            )
        return TransactionalRecord(table[record_id])

people = Table(DB, "Person")

assert len(people) == 8
assert 1 in people
assert 999 not in people
assert people(1).last_name == "Newton"

print(people.table_name, len(people))

Person 8


## Problem 10 — Persist a record back to the dictionary database

The previous `Table` returns a copy-like dynamic record, but `commit()` does not update `DB`.

Create `BoundRecord` that knows:

- which table mapping it belongs to;
- which record ID it represents.

Its `save()` method must copy current values back into the table and then mark them committed.

In [12]:
# SOLUTION 10

class BoundRecord(TransactionalRecord):
    def __init__(self, table_data: dict[int, dict[str, Any]], record_id: int):
        if record_id not in table_data:
            raise KeyError(record_id)
        super().__init__(table_data[record_id])
        object.__setattr__(self, "_table_data", table_data)
        object.__setattr__(self, "_record_id", record_id)

    @property
    def record_id(self) -> int:
        return object.__getattribute__(self, "_record_id")

    def save(self) -> None:
        table_data = object.__getattribute__(self, "_table_data")
        record_id = object.__getattribute__(self, "_record_id")
        table_data[record_id] = deepcopy(object.__getattribute__(self, "_record"))
        self.commit()

db_copy = deepcopy(DB)
bound = BoundRecord(db_copy["Person"], 1)
bound.last_name = "Newton-Smith"

assert db_copy["Person"][1]["last_name"] == "Newton"

bound.save()

assert db_copy["Person"][1]["last_name"] == "Newton-Smith"
assert bound.changes == {}
print(db_copy["Person"][1])

{'first_name': 'Isaac', 'last_name': 'Newton-Smith', 'born': 1642, 'country_id': 1}


## Problem 11 — Resolve foreign keys dynamically

For a field named `<target>_id`, expose a convenience relationship `<target>`.

Example:

```python
person.country_id  -> 1
person.country     -> BoundRecord for Country(1)
```

Use an explicit relationship map rather than guessing table names from arbitrary strings.

In [13]:
# SOLUTION 11

class Database:
    def __init__(self, data: dict[str, dict[int, dict[str, Any]]]):
        self._data = data
        self._relationships: dict[tuple[str, str], str] = {}

    def add_relationship(self, source_table: str, field: str, target_table: str) -> None:
        if source_table not in self._data:
            raise KeyError(source_table)
        if target_table not in self._data:
            raise KeyError(target_table)
        self._relationships[(source_table, field)] = target_table

    def table(self, table_name: str) -> "DatabaseTable":
        return DatabaseTable(self, table_name)

class DatabaseTable:
    def __init__(self, database: Database, table_name: str):
        if table_name not in database._data:
            raise KeyError(table_name)
        self._database = database
        self._table_name = table_name

    def __call__(self, record_id: int) -> "RelationalRecord":
        return RelationalRecord(self._database, self._table_name, record_id)

class RelationalRecord(BoundRecord):
    def __init__(self, database: Database, table_name: str, record_id: int):
        table_data = database._data[table_name]
        super().__init__(table_data, record_id)
        object.__setattr__(self, "_database", database)
        object.__setattr__(self, "_table_name", table_name)

    def __getattr__(self, name: str) -> Any:
        # 1) Normal dynamic fields first.
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]

        # 2) Then relationship aliases.
        database = object.__getattribute__(self, "_database")
        table_name = object.__getattribute__(self, "_table_name")
        fk_field = f"{name}_id"
        target_table = database._relationships.get((table_name, fk_field))

        if target_table is not None and fk_field in record:
            return database.table(target_table)(record[fk_field])

        raise AttributeError(name)

db_rel = Database(deepcopy(DB))
db_rel.add_relationship("Person", "country_id", "Country")

person = db_rel.table("Person")(1)
country = person.country

assert person.country_id == 1
assert country.name == "United Kingdom"
assert country.capital == "London"

print(person.first_name, "->", country.name, "/", country.capital)

Isaac -> United Kingdom / London


## Problem 12 — Add computed fields with clear precedence

Add computed fields such as `full_name` without storing them in the underlying record.

Precedence should be:

1. real Python attributes/methods/properties;
2. stored dynamic fields;
3. computed dynamic fields;
4. relationships;
5. `AttributeError`.

This keeps behavior predictable.

In [14]:
# SOLUTION 12

class ComputedRelationalRecord(RelationalRecord):
    COMPUTED_FIELDS: dict[str, Callable[["ComputedRelationalRecord"], Any]] = {}

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")

        # Stored dynamic field.
        if name in record:
            return record[name]

        # Computed dynamic field.
        computed = type(self).COMPUTED_FIELDS
        if name in computed:
            return computed[name](self)

        # Relationship fallback.
        return super().__getattr__(name)

class PersonRecord(ComputedRelationalRecord):
    COMPUTED_FIELDS = {
        "full_name": lambda self: f"{self.first_name} {self.last_name}",
        "age_in_2000": lambda self: 2000 - self.born,
    }

class TypedDatabaseTable(DatabaseTable):
    def __call__(self, record_id: int):
        if self._table_name == "Person":
            return PersonRecord(self._database, self._table_name, record_id)
        return ComputedRelationalRecord(self._database, self._table_name, record_id)

class TypedDatabase(Database):
    def table(self, table_name: str) -> TypedDatabaseTable:
        return TypedDatabaseTable(self, table_name)

typed_db = TypedDatabase(deepcopy(DB))
typed_db.add_relationship("Person", "country_id", "Country")

p = typed_db.table("Person")(3)

assert p.full_name == "Joseph Fourier"
assert p.age_in_2000 == 232
assert p.country.name == "France"

print(p.full_name, p.age_in_2000, p.country.name)

Joseph Fourier 232 France


## Problem 13 — Add safe serialization

Implement `to_dict()` with options:

- `include_computed=False` by default;
- optionally include selected computed fields;
- never leak internal attributes such as `_database`, `_record_id`, or `_dirty`.

In [15]:
# SOLUTION 13

class SerializablePersonRecord(PersonRecord):
    def to_dict(self, *, include_computed: bool = False) -> dict[str, Any]:
        data = deepcopy(object.__getattribute__(self, "_record"))

        if include_computed:
            for name in type(self).COMPUTED_FIELDS:
                data[name] = getattr(self, name)

        return data

class SerializableTable(TypedDatabaseTable):
    def __call__(self, record_id: int):
        if self._table_name == "Person":
            return SerializablePersonRecord(self._database, self._table_name, record_id)
        return ComputedRelationalRecord(self._database, self._table_name, record_id)

class SerializableDatabase(TypedDatabase):
    def table(self, table_name: str) -> SerializableTable:
        return SerializableTable(self, table_name)

sdb = SerializableDatabase(deepcopy(DB))
sdb.add_relationship("Person", "country_id", "Country")

person = sdb.table("Person")(2)

plain = person.to_dict()
enriched = person.to_dict(include_computed=True)

assert "_database" not in plain
assert "full_name" not in plain
assert enriched["full_name"] == "Gottfried von Leibniz"

print(json.dumps(enriched, indent=2))

{
  "first_name": "Gottfried",
  "last_name": "von Leibniz",
  "born": 1646,
  "country_id": 5,
  "full_name": "Gottfried von Leibniz",
  "age_in_2000": 354
}


## Problem 14 — Cache relationship lookups and invalidate correctly

Relationship access may repeatedly create new wrapper objects. Add a small per-record cache.

Requirements:

- repeated `person.country` access returns the same wrapper object;
- changing `country_id` invalidates cached `country`;
- normal fields do not invalidate unrelated cached relationships.

In [16]:
# SOLUTION 14

class CachedRelationalRecord(RelationalRecord):
    def __init__(self, database: Database, table_name: str, record_id: int):
        super().__init__(database, table_name, record_id)
        object.__setattr__(self, "_relation_cache", {})

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]

        cache = object.__getattribute__(self, "_relation_cache")
        if name in cache:
            return cache[name]

        database = object.__getattribute__(self, "_database")
        table_name = object.__getattribute__(self, "_table_name")
        fk_field = f"{name}_id"
        target_table = database._relationships.get((table_name, fk_field))

        if target_table is not None and fk_field in record:
            related = database.table(target_table)(record[fk_field])
            cache[name] = related
            return related

        raise AttributeError(name)

    def __setattr__(self, name: str, value: Any) -> None:
        super().__setattr__(name, value)

        if name.startswith("_"):
            return

        database = object.__getattribute__(self, "_database")
        table_name = object.__getattribute__(self, "_table_name")
        cache = object.__getattribute__(self, "_relation_cache")

        for (source_table, fk_field), target_table in database._relationships.items():
            if source_table == table_name and fk_field == name and name.endswith("_id"):
                relation_name = name[:-3]
                cache.pop(relation_name, None)

class CachedTable(DatabaseTable):
    def __call__(self, record_id: int):
        return CachedRelationalRecord(self._database, self._table_name, record_id)

class CachedDatabase(Database):
    def table(self, table_name: str) -> CachedTable:
        return CachedTable(self, table_name)

cdb = CachedDatabase(deepcopy(DB))
cdb.add_relationship("Person", "country_id", "Country")

p = cdb.table("Person")(1)

first_country = p.country
second_country = p.country
assert first_country is second_country

p.country_id = 5
third_country = p.country

assert third_country is not first_country
assert third_country.name == "Germany"

print(first_country.name, "->", third_country.name)

United Kingdom -> Germany


## Advanced debugging example — `__getattribute__` is powerful but dangerous

This example is intentionally small. It shows how to instrument *every* attribute access while avoiding recursion.

Notice that the implementation uses `object.__getattribute__` internally rather than `self.__dict__` or `self.some_name`.

In [17]:
class AccessLogger:
    def __init__(self, value):
        object.__setattr__(self, "_value", value)

    def __getattribute__(self, name):
        # Avoid using self.<anything> here unless you intentionally want recursion.
        if name != "_log":
            object.__getattribute__(self, "_log")(name)
        return object.__getattribute__(self, name)

    def _log(self, name):
        print(f"access -> {name}")

    @property
    def doubled(self):
        return self._value * 2

logger = AccessLogger(21)
assert logger.doubled == 42

access -> doubled
access -> _value


## Problem 15 — Implement a guarded `__getattribute__`

Create a class that blocks access to a field called `secret` while leaving all other attributes unchanged.

This is deliberately a `__getattribute__` exercise because the goal is to intercept even an existing attribute.

Do **not** use this pattern for ordinary dynamic-field fallback.

In [18]:
# SOLUTION 15

class Guarded:
    def __init__(self, public, secret):
        object.__setattr__(self, "public", public)
        object.__setattr__(self, "secret", secret)

    def __getattribute__(self, name):
        if name == "secret":
            raise AttributeError("secret is not exposed")
        return object.__getattribute__(self, name)

g = Guarded("visible", "hidden")

assert g.public == "visible"
assert not hasattr(g, "secret")
print(g.public)

visible


## Problem 16 — Prevent accidental mutation through shared nested objects

A shallow `dict(record)` copy does not protect nested lists/dicts.

Demonstrate the bug, then create `DeepIsolatedRecord` that deep-copies input data and returns defensive copies for mutable dynamic values.

In [19]:
# SOLUTION 16

class DeepIsolatedRecord:
    def __init__(self, record: Mapping[str, Any]):
        object.__setattr__(self, "_record", deepcopy(dict(record)))

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise AttributeError(name)
        return deepcopy(record[name])

    def __setattr__(self, name: str, value: Any) -> None:
        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise AttributeError(name)
        record[name] = deepcopy(value)

original = {"name": "sample", "tags": ["math", "python"]}
isolated = DeepIsolatedRecord(original)

tags = isolated.tags
tags.append("mutated elsewhere")

assert isolated.tags == ["math", "python"]
assert original["tags"] == ["math", "python"]

print(isolated.tags)

['math', 'python']


## Problem 17 — Make records immutable after construction

Implement `FrozenRecord`:

- dynamic fields are readable;
- any assignment to an existing or unknown public field fails;
- internal bootstrap still works;
- error messages are explicit.

In [20]:
# SOLUTION 17

class FrozenRecord:
    def __init__(self, record: Mapping[str, Any]):
        object.__setattr__(self, "_record", deepcopy(dict(record)))

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]
        raise AttributeError(name)

    def __setattr__(self, name: str, value: Any) -> None:
        raise AttributeError(f"{type(self).__name__} is immutable; cannot assign {name!r}")

frozen = FrozenRecord({"x": 1})
assert frozen.x == 1

try:
    frozen.x = 2
except AttributeError as exc:
    print(exc)

FrozenRecord is immutable; cannot assign 'x'


## Problem 18 — Add mapping-style access alongside attribute access

Attribute syntax cannot naturally represent arbitrary keys such as `"first-name"`, `"class"`, or `"2fa_enabled"`.

Build a record that supports both:

```python
record.first_name
record["first-name"]
```

Use mapping access as the universal escape hatch.

In [21]:
# SOLUTION 18

class HybridRecord(ManagedRecord):
    def __getitem__(self, key: str) -> Any:
        record = object.__getattribute__(self, "_record")
        return record[key]

    def __setitem__(self, key: str, value: Any) -> None:
        record = object.__getattribute__(self, "_record")
        if key not in record:
            raise KeyError(key)
        old = record[key]
        record[key] = value
        if old != value:
            object.__getattribute__(self, "_dirty").add(key)

    def __iter__(self):
        return iter(object.__getattribute__(self, "_record"))

    def __len__(self):
        return len(object.__getattribute__(self, "_record"))

hybrid = HybridRecord({
    "first_name": "Grace",
    "first-name": "Grace-via-weird-key",
    "2fa_enabled": True,
})

assert hybrid.first_name == "Grace"
assert hybrid["first-name"] == "Grace-via-weird-key"
assert hybrid["2fa_enabled"] is True

print(dict((key, hybrid[key]) for key in hybrid))

{'first_name': 'Grace', 'first-name': 'Grace-via-weird-key', '2fa_enabled': True}


# Capstone — Mini ORM

Combine the strongest ideas into one compact design:

- safe internal bootstrap;
- dynamic fields;
- validation;
- dirty tracking;
- save/rollback;
- relationships;
- computed fields;
- `__dir__`;
- mapping fallback;
- serialization.

The goal is not to replace SQLAlchemy or Django ORM. The goal is to exercise Python's attribute protocol correctly.

In [22]:
@dataclass(frozen=True)
class FieldSpec:
    validator: Callable[[Any], None] | None = None

@dataclass(frozen=True)
class RelationSpec:
    fk_field: str
    target_table: str

class MiniORM:
    def __init__(self, data: dict[str, dict[int, dict[str, Any]]]):
        self.data = data
        self.schemas: dict[str, dict[str, FieldSpec]] = {}
        self.relations: dict[str, dict[str, RelationSpec]] = {}
        self.computed: dict[str, dict[str, Callable[["ORMRecord"], Any]]] = {}

    def register_schema(self, table: str, schema: dict[str, FieldSpec]) -> None:
        self.schemas[table] = dict(schema)

    def register_relation(
        self,
        table: str,
        relation_name: str,
        fk_field: str,
        target_table: str,
    ) -> None:
        self.relations.setdefault(table, {})[relation_name] = RelationSpec(
            fk_field=fk_field,
            target_table=target_table,
        )

    def register_computed(
        self,
        table: str,
        name: str,
        func: Callable[["ORMRecord"], Any],
    ) -> None:
        self.computed.setdefault(table, {})[name] = func

    def table(self, name: str) -> "ORMTable":
        if name not in self.data:
            raise KeyError(f"Unknown table {name!r}")
        return ORMTable(self, name)

class ORMTable:
    def __init__(self, orm: MiniORM, name: str):
        self.orm = orm
        self.name = name

    def __call__(self, record_id: int) -> "ORMRecord":
        return ORMRecord(self.orm, self.name, record_id)

    def all(self) -> list["ORMRecord"]:
        return [self(record_id) for record_id in sorted(self.orm.data[self.name])]

class ORMRecord:
    def __init__(self, orm: MiniORM, table_name: str, record_id: int):
        table_data = orm.data[table_name]
        if record_id not in table_data:
            raise KeyError(
                f"Record {record_id!r} not found in table {table_name!r}"
            )

        object.__setattr__(self, "_orm", orm)
        object.__setattr__(self, "_table_name", table_name)
        object.__setattr__(self, "_record_id", record_id)
        object.__setattr__(self, "_record", deepcopy(table_data[record_id]))
        object.__setattr__(self, "_original", deepcopy(table_data[record_id]))
        object.__setattr__(self, "_relation_cache", {})

    @property
    def id(self) -> int:
        return object.__getattribute__(self, "_record_id")

    @property
    def changes(self) -> dict[str, tuple[Any, Any]]:
        record = object.__getattribute__(self, "_record")
        original = object.__getattribute__(self, "_original")
        return {
            name: (original.get(name), value)
            for name, value in record.items()
            if original.get(name) != value
        }

    def __getattr__(self, name: str) -> Any:
        record = object.__getattribute__(self, "_record")
        if name in record:
            return record[name]

        orm = object.__getattribute__(self, "_orm")
        table_name = object.__getattribute__(self, "_table_name")

        computed_for_table = orm.computed.get(table_name, {})
        if name in computed_for_table:
            return computed_for_table[name](self)

        relations_for_table = orm.relations.get(table_name, {})
        if name in relations_for_table:
            cache = object.__getattribute__(self, "_relation_cache")
            if name not in cache:
                spec = relations_for_table[name]
                fk_value = record[spec.fk_field]
                cache[name] = orm.table(spec.target_table)(fk_value)
            return cache[name]

        raise AttributeError(
            f"{type(self).__name__} has no attribute or dynamic field {name!r}"
        )

    def __setattr__(self, name: str, value: Any) -> None:
        if name.startswith("_"):
            object.__setattr__(self, name, value)
            return

        record = object.__getattribute__(self, "_record")
        if name not in record:
            raise AttributeError(f"Unknown stored field {name!r}")

        orm = object.__getattribute__(self, "_orm")
        table_name = object.__getattribute__(self, "_table_name")
        spec = orm.schemas.get(table_name, {}).get(name)

        if spec is not None and spec.validator is not None:
            spec.validator(value)

        old_value = record[name]
        record[name] = deepcopy(value)

        if old_value != value:
            relations_for_table = orm.relations.get(table_name, {})
            cache = object.__getattribute__(self, "_relation_cache")
            for relation_name, relation_spec in relations_for_table.items():
                if relation_spec.fk_field == name:
                    cache.pop(relation_name, None)

    def __getitem__(self, key: str) -> Any:
        return object.__getattribute__(self, "_record")[key]

    def __setitem__(self, key: str, value: Any) -> None:
        setattr(self, key, value)

    def __dir__(self):
        names = set(super().__dir__())
        record = object.__getattribute__(self, "_record")
        orm = object.__getattribute__(self, "_orm")
        table_name = object.__getattribute__(self, "_table_name")

        names.update(record)
        names.update(orm.computed.get(table_name, {}))
        names.update(orm.relations.get(table_name, {}))
        return sorted(names)

    def save(self) -> None:
        orm = object.__getattribute__(self, "_orm")
        table_name = object.__getattribute__(self, "_table_name")
        record_id = object.__getattribute__(self, "_record_id")
        record = object.__getattribute__(self, "_record")

        orm.data[table_name][record_id] = deepcopy(record)
        object.__setattr__(self, "_original", deepcopy(record))

    def rollback(self) -> None:
        original = object.__getattribute__(self, "_original")
        object.__setattr__(self, "_record", deepcopy(original))
        object.__getattribute__(self, "_relation_cache").clear()

    def to_dict(self, *, include_computed: bool = False) -> dict[str, Any]:
        data = deepcopy(object.__getattribute__(self, "_record"))

        if include_computed:
            orm = object.__getattribute__(self, "_orm")
            table_name = object.__getattribute__(self, "_table_name")
            for name in orm.computed.get(table_name, {}):
                data[name] = getattr(self, name)

        return data

    def __repr__(self) -> str:
        table_name = object.__getattribute__(self, "_table_name")
        record_id = object.__getattribute__(self, "_record_id")
        return f"<ORMRecord {table_name} id={record_id}>"

In [23]:
# Configure and exercise the capstone ORM.

capstone_db = deepcopy(DB)
orm = MiniORM(capstone_db)

orm.register_schema(
    "Person",
    {
        "first_name": FieldSpec(validate_non_empty_string),
        "last_name": FieldSpec(validate_non_empty_string),
        "born": FieldSpec(validate_year),
        "country_id": FieldSpec(validate_positive_int),
    },
)

orm.register_relation(
    table="Person",
    relation_name="country",
    fk_field="country_id",
    target_table="Country",
)

orm.register_computed(
    "Person",
    "full_name",
    lambda person: f"{person.first_name} {person.last_name}",
)

people = orm.table("Person")
person = people(1)

assert person.first_name == "Isaac"
assert person.full_name == "Isaac Newton"
assert person.country.capital == "London"
assert "full_name" in dir(person)
assert "country" in dir(person)

print(person)
print(person.to_dict(include_computed=True))

<ORMRecord Person id=1>
{'first_name': 'Isaac', 'last_name': 'Newton', 'born': 1642, 'country_id': 1, 'full_name': 'Isaac Newton'}


In [24]:
# Mutation, validation, relationship-cache invalidation, rollback, and persistence.

person = people(1)

original_country = person.country
assert original_country.name == "United Kingdom"

person.country_id = 5
assert person.country.name == "Germany"
assert person.changes["country_id"] == (1, 5)

person.last_name = "Newton (edited)"
assert person.full_name == "Isaac Newton (edited)"

# Roll back unsaved changes.
person.rollback()
assert person.country_id == 1
assert person.last_name == "Newton"
assert person.country.name == "United Kingdom"

# Make and persist a different change.
person.last_name = "Newton-Smith"
person.save()

assert capstone_db["Person"][1]["last_name"] == "Newton-Smith"
assert person.changes == {}

print("Persisted:", capstone_db["Person"][1])

Persisted: {'first_name': 'Isaac', 'last_name': 'Newton-Smith', 'born': 1642, 'country_id': 1}


# Challenge set — Try before reading the hints

These do not introduce new machinery, but they force you to reason about precedence, error semantics, introspection, and state consistency.

### Challenge A — Deletion semantics

Implement `__delattr__` so only nullable fields may be deleted. Decide whether deletion means:

- remove the key entirely, or
- keep the key and assign `None`.

Document your contract.

### Challenge B — Read-only schema fields

Extend `FieldSpec` with `read_only: bool = False`. Prevent writes to read-only fields after construction.

### Challenge C — Defaults

When a schema contains a field missing from a record, allow a default value or default factory.

### Challenge D — Optimistic concurrency

Track a record version. Make `save()` fail if another writer changed the underlying version since this wrapper was created.

### Challenge E — Batch transaction

Stage changes across multiple records and either persist all of them or roll all of them back.

### Challenge F — Descriptor comparison

Rebuild one field using a descriptor instead of `__getattr__`. Compare:

- IDE discoverability,
- static typing,
- runtime flexibility,
- repetitive code,
- validation placement.

## Challenge hints

- For `__delattr__`, avoid `del self.some_field` inside the hook; mutate `_record` directly.
- For read-only fields, schema lookup belongs before mutation.
- Defaults can be modeled with a sentinel object so `None` remains a valid default.
- Optimistic concurrency usually requires a version stored in the backing record.
- A batch transaction should validate all pending writes before applying any of them.
- Descriptors shine when the field set is known at class-definition time; `__getattr__` shines when the field set is dynamic.

# Best-practice checklist

1. Prefer `__getattr__` over `__getattribute__` for fallback behavior.
2. Use `object.__setattr__` / `object.__getattribute__` for trusted internal access inside hooks.
3. Raise `AttributeError` for missing attributes.
4. Keep special methods small; delegate validation and persistence to normal helpers.
5. Separate internal names from public dynamic fields.
6. Provide an explicit mapping or `get_field` API for collision-prone/arbitrary field names.
7. Consider `__dir__` for interactive discoverability.
8. Define precedence rules for stored fields, computed fields, methods, and relationships.
9. Deep-copy when aliasing mutable data would violate your abstraction.
10. Invalidate caches when their dependencies change.
11. Keep persistence semantics explicit: local mutation, save/commit, rollback.
12. Add tests for recursion, unknown attributes, collisions, validation failure, rollback, and cache invalidation.
13. Do not use magic methods merely to make an API look clever; use them when they make semantics clearer.

# Final self-test

Run the cell below. If it prints `ALL SELF-TESTS PASSED`, the notebook's major examples are behaving consistently.

In [25]:
def run_self_tests():
    # Read-only fallback
    ro = ReadOnlyRecord({"a": 1})
    assert ro.a == 1
    assert not hasattr(ro, "missing")

    # Managed writes
    m = ManagedRecord({"x": 1})
    m.x = 2
    assert m.x == 2
    assert m._dirty == {"x"}

    # Validation
    vr = ValidatedRecord(
        {"first_name": "A", "last_name": "B", "born": 2000, "country_id": 1},
        PERSON_SCHEMA,
    )
    try:
        vr.born = "bad"
    except TypeError:
        pass
    else:
        raise AssertionError("invalid year should fail")

    # Transaction
    tr = TransactionalRecord({"x": 1})
    tr.x = 2
    tr.rollback()
    assert tr.x == 1

    # Relationship
    local_db = Database(deepcopy(DB))
    local_db.add_relationship("Person", "country_id", "Country")
    assert local_db.table("Person")(1).country.name == "United Kingdom"

    # Capstone persistence
    local_data = deepcopy(DB)
    local_orm = MiniORM(local_data)
    local_orm.register_schema("Person", {
        "first_name": FieldSpec(validate_non_empty_string),
        "last_name": FieldSpec(validate_non_empty_string),
        "born": FieldSpec(validate_year),
        "country_id": FieldSpec(validate_positive_int),
    })
    local_orm.register_relation("Person", "country", "country_id", "Country")
    local_orm.register_computed("Person", "full_name",
                                lambda p: f"{p.first_name} {p.last_name}")

    q = local_orm.table("Person")(1)
    assert q.full_name == "Isaac Newton"
    assert q.country.capital == "London"
    q.last_name = "N."
    assert q.changes["last_name"] == ("Newton", "N.")
    q.save()
    assert local_data["Person"][1]["last_name"] == "N."

    print("ALL SELF-TESTS PASSED")

run_self_tests()

ALL SELF-TESTS PASSED
